In [1]:
pip install pycountry-convert


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install countryinfo

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import re
import os
import pycountry_convert as pc
from countryinfo import CountryInfo

def module_2_comprehensive_clean(input_file="military_raw_.csv", output_file="military_cleaned.csv"):
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    df = pd.read_csv(input_file)

    # 1. Standardize and Rename
    df.columns = [c.lower().replace("-", "_") for c in df.columns]
    
    rename_map = {
        'total_population_by_country': 'population',
        'active_military_manpower': 'active_personnel',
        'active_reserve_military_manpower': 'reserve_personnel',
        'defense_spending_budget': 'defense_budget_usd',
        'purchasing_power_parity': 'gdp_usd',
        'square_land_area': 'land_area_sq_km',
        'coastline_coverage': 'coastline_km',
        'aircraft_total': 'total_aircraft',
        'aircraft_total_fighters': 'fighter_aircraft',
        'aircraft_total_attack_types': 'attack_aircraft',
        'aircraft_total_transports': 'transport_aircraft',
        'aircraft_helicopters_total': 'helicopters',
        'armor_tanks_total': 'tanks',
        'armor_apc_total': 'armored_vehicles',
        'armor_self_propelled_guns_total': 'artillery_units', 
        'navy_ships': 'naval_assets',
        'navy_aircraft_carriers': 'aircraft_carriers',
        'pwrind_score': 'Power_index_score',
        'pwrind_rank': 'power_index_rank'
    }
    df = df.rename(columns=rename_map)

    # 2. Super Cleaner
    def super_clean(val, col_name):
        if pd.isna(val) or str(val).strip().lower() in ['n/a', '', 'nan', '--']:
            return 0.0
        val_str = str(val).replace('\n', '').replace('\r', '').strip()
        cleaned = re.sub(r'[^\d.]', '', val_str)
        try:
            if "score" in col_name.lower() or "." in cleaned:
                return float(cleaned)
            return int(cleaned) if cleaned else 0
        except:
            return 0.0

    for col in df.columns:
        if col != "country":
            df[col] = df.apply(lambda row: super_clean(row[col], col), axis=1)

    # 3. Geographical Mapping (auto via pycountry_convert)
    def get_continent(country_name):
        try:
            country_alpha2 = pc.country_name_to_country_alpha2(country_name)
            continent_code = pc.country_alpha2_to_continent_code(country_alpha2)
            continent_map = {
                "AF": "Africa",
                "AS": "Asia",
                "EU": "Europe",
                "NA": "North America",
                "SA": "South America",
                "OC": "Oceania"
            }
            return continent_map.get(continent_code, "Unknown")
        except:
            return "Unknown"

    df["continent"] = df["country"].apply(get_continent)
    df["region"] = df["continent"]

    # 4. Capital Mapping (hybrid: auto + manual fallback)
    capital_data_manual = {
        "Turkiye": "Ankara",
        "Turkey": "Ankara",
        "United States": "Washington, D.C.",
        "Russia": "Moscow",
        "China": "Beijing",
        "India": "New Delhi"
        # Add more tricky cases if needed
    }

    def get_capital(country_name):
        if country_name in capital_data_manual:
            return capital_data_manual[country_name]
        try:
            info = CountryInfo(country_name)
            return info.capital()
        except:
            return "Unknown"

    df["capital_city"] = df["country"].apply(get_capital)

    # 5. Alliance Mapping
    nato_list = ["United States", "United Kingdom", "France", "Germany", "Italy", "Poland", "Canada", "Norway", "Netherlands", "Spain", "Turkiye", "Turkey", "Greece"]
    non_nato_list = ["Russia", "China", "India", "Brazil", "South Africa", "Mexico", "Japan"]  # Example list

    df["alliance"] = df["country"].apply(
        lambda x: "NATO" if x in nato_list 
        else "Non-NATO" if x in non_nato_list 
        else "Others"
    )

    df["year"] = 2025
    
    # Calculate Total Personnel
    df['total_personnel'] = df.get('active_personnel', 0) + df.get('reserve_personnel', 0)

    # 6. Final Filtering (Expanded list)
    pdf_cols_only = [
        "country", "capital_city", "region", "continent", "alliance", "year",
        "population", "defense_budget_usd", "gdp_usd", "land_area_sq_km", "coastline_km",
        "active_personnel", "reserve_personnel", "total_personnel",
        "total_aircraft", "fighter_aircraft", "attack_aircraft", "transport_aircraft",
        "helicopters", "tanks", "armored_vehicles", "artillery_units",
        "naval_assets", "aircraft_carriers",
        "Power_index_score", "power_index_rank"
    ]

    df = df[pdf_cols_only]

    # Save cleaned file
    df.to_csv(output_file, index=False)
    print(f"Cleaned data saved to {output_file}")

# Run function
module_2_comprehensive_clean()

Cleaned data saved to military_cleaned.csv
